<a href="https://colab.research.google.com/github/shadman-shakib/AI-and-Visual-Intelligence/blob/main/finalproject_AGST892_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# Choose where inside Drive you want the project to live
# (edit this line if you want a different folder)
PROJECT_DIR = "/content/drive/MyDrive/cattle-id"

# Create folders directly in Drive
from pathlib import Path
PROJ = Path(PROJECT_DIR)
PROJ.mkdir(parents=True, exist_ok=True)

# (Optional) make subfolders for clarity
for sub in ["data", "runs", "ocr"]:
    (PROJ/sub).mkdir(parents=True, exist_ok=True)

print("✅ All files will be saved permanently in:", PROJ)


✅ All files will be saved permanently in: /content/drive/MyDrive/cattle-id


In [ ]:
!uv pip install ultralytics
import ultralytics  # https://www.ultralytics.com/
ultralytics.checks()

Ultralytics 8.3.225 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
Setup complete ✅ (12 CPUs, 167.1 GB RAM, 39.3/235.7 GB disk)


In [ ]:
#@title 2️⃣ Link your YOLOv11 dataset from Google Drive
DATASET_DIR = "/content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/"  # <-- change if needed
DATASET_DIR


'/content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/'

In [ ]:
#@title Check dataset structure
import os
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in files[:5]:
        print(f"{subindent}{f}")


/
  data.yaml
  README.dataset.txt
  README.roboflow.txt
train/
  labels.cache
  labels/
    DSCF8510_JPG.rf.a3d319df93a00239a86373b5796174a1.txt
    DSCF8705_JPG.rf.186f7c18b47c0b3009a21a34d01fa087.txt
    DSCF8737_JPG.rf.40484df20008deb7415cc8f44ccf9052.txt
    DSCF8566_JPG.rf.5502ebb37cdc972aaa44bee230f7960d.txt
    DSCF8521_JPG.rf.d81c94e7b2227a94c8c15719a9f0a1e8.txt
  images/
    DSCF8052_JPG.rf.3412cbc143400a6860e9ac2f24e775d9.jpg
    DSCF8198_JPG.rf.f35b59e55e3b871f5a833e2d9fe83e73.jpg
    DSCF8589_JPG.rf.b029bce33a517a29d2e55930a1986e79.jpg
    DSCF8274_JPG.rf.2ec129a14162ad4d840febec1117a271.jpg
    DSCF8139_JPG.rf.954df015ff286a36674bd72225a80a83.jpg
test/
  labels/
    DSCF0813_JPG.rf.decf94fdbded2e90905e4c9e9aa7ac1d.txt
    DSCF0783_JPG.rf.ccfabb543e30e92cb73801534e316f8c.txt
    DSCF0818_JPG.rf.98af0110423d3fc4c63936d4b1c30e25.txt
    DSCF0805_JPG.rf.d1b59eff9c0126a2bf1219d3f1ccddaf.txt
    DSCF0826_JPG.rf.c3a649ac24aaa56b1b79099fa2da10ee.txt
  images/
    DSCF0866_JPG.rf.

In [ ]:
#@title 3️⃣ Copy YAML file and preview contents
from shutil import copyfile
from pathlib import Path

yaml_src = Path(DATASET_DIR)/"data.yaml"
yaml_dst = PROJ/"data.yaml"
copyfile(yaml_src, yaml_dst)
print("✅ Copied data.yaml to:", yaml_dst)
print("\n---- YAML Content Before Update ----")
print(yaml_dst.read_text())

# Update the data.yaml file with correct paths
with open(yaml_dst, 'r') as f:
    lines = f.readlines()

with open(yaml_dst, 'w') as f:
    for line in lines:
        if line.strip().startswith('train:'):
            f.write(f"train: {DATASET_DIR}/train/images\n")
        elif line.strip().startswith('val:'):
            f.write(f"val: {DATASET_DIR}/valid/images\n")
        elif line.strip().startswith('test:'):
            f.write(f"test: {DATASET_DIR}/test/images\n")
        else:
            f.write(line)

print("\n---- YAML Content After Update ----")
print(yaml_dst.read_text())

✅ Copied data.yaml to: /content/drive/MyDrive/cattle-id/data.yaml

---- YAML Content Before Update ----
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 4
names: ['Cattle', 'Cattle Ear Tag', 'Cattle Head Whorl', 'Cattle Muzzle']

roboflow:
  workspace: gradsjunior4
  project: csce873cv
  version: 4
  license: CC BY 4.0
  url: https://universe.roboflow.com/gradsjunior4/csce873cv/dataset/4

---- YAML Content After Update ----
train: /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11//train/images
val: /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11//valid/images
test: /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11//test/images

nc: 4
names: ['Cattle', 'Cattle Ear Tag', 'Cattle Head Whorl', 'Cattle Muzzle']

roboflow:
  workspace: gradsjunior4
  project: csce873cv
  version: 4
  license: CC BY 4.0
  url: https://universe.roboflow.com/gradsjunior4/csce873cv/dataset/4


In [ ]:
#@title 4️⃣ Train YOLOv11x model
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # smaller pretrained model
results = model.train(
    data=str(yaml_dst),
    epochs=100,
    imgsz=640, # Reduced image size
    batch=2,  # Further reduced batch size
    device=0,                # use GPU
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=1e-4,
    cos_lr=True,
    amp=True,
    patience=30,
    mosaic=1.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    translate=0.1, scale=0.5, shear=0.1, flipud=0.0, fliplr=0.5,
    auto_augment="randaugment",
    workers=2,
    project=str(PROJ/"runs"), name="yolo11n_train" # Changed project name to reflect model change
)
print("✅ Training done. Best weights located at:")
print(PROJ/"runs/detect/yolo11n_train/weights/best.pt") # Changed path to reflect model change

Ultralytics 8.3.225 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/cattle-id/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n_train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, pe

In [ ]:
#@title 🧪 Evaluate on test split
from ultralytics import YOLO
best_w = "/content/drive/MyDrive/cattle-id/runs/yolo11n_train3/weights/best.pt"
m = YOLO(str(best_w))
m.val(data=yaml_path, split="test")


Ultralytics 8.3.225 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 0.0±0.0 MB/s, size: 47.7 KB)
val: Scanning /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/labels... 337 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 337/337 8.0it/s 42.4s
val: New cache created: /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 4.7it/s 4.7s
                   all        337       1320      0.925       0.96      0.978      0.702
                Cattle        337        337      0.947      0.997      0.995      0.842
        Cattle Ear Tag        322        322      0.992          1      0.995      0.697
     Cattle Head Whorl        324        324      0.935      0.983      0.988      0.617
         Cat

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7936a1d38f80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
#@title 🖼️ Save predicted images to Drive
pred_dir = PROJ/"runs/pred_test"
PROJECT_DIR = "/content/drive/MyDrive/cattle-id"
m.predict(
    source=str(Path(DATASET_DIR)/"test/images"),
    conf=0.25,
    imgsz=640,
    save=True,
    project=str(PROJ/"runs"),
    name="pred_test"
)
print("✅ Predictions saved to:", pred_dir)



image 1/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0745_JPG.rf.b93586a9744b66b8d3d58add1305f6fc.jpg: 640x640 1 Cattle, 2 Cattle Ear Tags, 1 Cattle Head Whorl, 1 Cattle Muzzle, 10.3ms
image 2/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0758_JPG.rf.7ef008cd9dddbc490a9b205163504e8b.jpg: 640x640 1 Cattle, 1 Cattle Ear Tag, 1 Cattle Head Whorl, 1 Cattle Muzzle, 17.0ms
image 3/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0759_JPG.rf.49cb06e0b9686e42d9934280e09b623c.jpg: 640x640 1 Cattle, 1 Cattle Ear Tag, 1 Cattle Head Whorl, 1 Cattle Muzzle, 9.6ms
image 4/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0761_JPG.rf.cf966300f2721bb9b450e2ab9687ac24.jpg: 640x640 1 Cattle, 1 Cattle Ear Tag, 1 Cattle Head Whorl, 2 Cattle Muzzles, 13.7ms
image 5/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0762_JPG.rf.1639e7102573887b98427722ced0120e.jpg: 640x640 1

In [ ]:
#@title ✂️ Crop ear-tag & muzzle crops for OCR / re-ID
from ultralytics import YOLO
import cv2
from pathlib import Path

out_ear = PROJ/"ocr/crops/ear-tag"; out_ear.mkdir(parents=True, exist_ok=True)
out_muz = PROJ/"ocr/crops/muzzle";  out_muz.mkdir(parents=True, exist_ok=True)

preds = m.predict(source=str(Path(DATASET_DIR)/"test/images"), conf=0.25, imgsz=640, save=False, stream=True)

num_ear = num_muz = 0
for r in preds:
    im = r.orig_img; h, w = im.shape[:2]; base = Path(r.path).stem
    for b in r.boxes:
        cls = int(b.cls.item())
        x1,y1,x2,y2 = map(int, b.xyxy[0].tolist())
        crop = im[max(0,y1):min(h,y2), max(0,x1):min(w,x2)]
        if cls == 2:  # ear-tag
            cv2.imwrite(str(out_ear/f"{base}_{x1}_{y1}.jpg"), crop); num_ear += 1
        elif cls == 1:  # muzzle
            cv2.imwrite(str(out_muz/f"{base}_{x1}_{y1}.jpg"), crop); num_muz += 1

print(f"✅ Saved: {num_ear} ear-tag crops → {out_ear}")
print(f"✅ Saved: {num_muz} muzzle crops → {out_muz}")



image 1/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0745_JPG.rf.b93586a9744b66b8d3d58add1305f6fc.jpg: 640x640 1 Cattle, 2 Cattle Ear Tags, 1 Cattle Head Whorl, 1 Cattle Muzzle, 10.9ms
image 2/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0758_JPG.rf.7ef008cd9dddbc490a9b205163504e8b.jpg: 640x640 1 Cattle, 1 Cattle Ear Tag, 1 Cattle Head Whorl, 1 Cattle Muzzle, 9.5ms
image 3/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0759_JPG.rf.49cb06e0b9686e42d9934280e09b623c.jpg: 640x640 1 Cattle, 1 Cattle Ear Tag, 1 Cattle Head Whorl, 1 Cattle Muzzle, 9.3ms
image 4/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0761_JPG.rf.cf966300f2721bb9b450e2ab9687ac24.jpg: 640x640 1 Cattle, 1 Cattle Ear Tag, 1 Cattle Head Whorl, 2 Cattle Muzzles, 9.6ms
image 5/337 /content/drive/MyDrive/cattle-id/CSCE873CV.v4i.yolov11/test/images/DSCF0762_JPG.rf.1639e7102573887b98427722ced0120e.jpg: 640x640 1 C

In [ ]:
#@title 🔎 OCR ear-tags with EasyOCR
import easyocr, re, json
reader = easyocr.Reader(['en'], gpu=True)

ocr_out = {}
for p in sorted((PROJ/"ocr/crops/muzzle").glob("*.jpg")):
    txts = reader.readtext(str(p), detail=0)
    tokens = [re.sub(r"[^0-9A-Za-z\-]", "", t) for t in txts]
    tokens = [t for t in tokens if len(t) >= 2]
    pred = max(tokens, key=len) if tokens else ""
    ocr_out[p.stem] = {"ocr": pred, "raw": txts}

ocr_json = PROJ/"ocr/ear_tag_ocr.json"
ocr_json.write_text(json.dumps(ocr_out, indent=2))
print("✅ OCR saved at:", ocr_json, "| total:", len(ocr_out))


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✅ OCR saved at: /content/drive/MyDrive/cattle-id/ocr/ear_tag_ocr.json | total: 334


In [ ]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 33.0 MB/s eta 0:00:00


In [ ]:
#@title 🐮 (Optional) Train a simple muzzle classifier
from pathlib import Path
DATA_DIR = PROJ/"ocr/crops/ear-tag"  # expects subfolders per ID: muzzle/<ID>/*.jpg

if len(list(DATA_DIR.glob("*/*"))) == 0:
    print("⏭️ Skipping: organize muzzle crops into subfolders per identity first.")
else:
    import torch, torch.nn as nn, torch.optim as optim
    from torchvision import models, transforms, datasets
    from torch.utils.data import DataLoader

    bs, sz = 32, 224
    tfm = transforms.Compose([
        transforms.Resize((sz, sz)),
        transforms.ColorJitter(0.1,0.1,0.1,0.05),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    ds = datasets.ImageFolder(str(DATA_DIR), transform=tfm)
    dl = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=2)
    num_classes = len(ds.classes)

    backbone = models.mobilenet_v3_large(weights="DEFAULT")
    backbone.classifier[-1] = nn.Linear(backbone.classifier[-1].in_features, num_classes)
    model = backbone.to("cuda" if torch.cuda.is_available() else "cpu")

    criterion = nn.CrossEntropyLoss(); opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    for epoch in range(10):
        model.train()
        tot, correct, n = 0, 0, 0
        for x,y in dl:
            x,y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*x.size(0); correct += (out.argmax(1)==y).sum().item(); n += x.size(0)
        print(f"epoch {epoch+1}: loss={tot/n:.4f} acc={correct/n:.3f}")

    MUZZLE_W = PROJ/"ocr/muzzle_cls.pt"
    import torch as _torch
    _torch.save(model.state_dict(), MUZZLE_W)
    print("✅ Saved muzzle classifier to:", MUZZLE_W)


⏭️ Skipping: organize muzzle crops into subfolders per identity first.


In [ ]:
#@title 🧪 (Optional) Fuse OCR + muzzle and compute metrics
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

CSV_PATH = PROJ/"eval/fused_predictions.csv"  # <-- put your CSV here
if not CSV_PATH.exists():
    print("⏭️ Provide fused_predictions.csv to compute metrics.")
else:
    df = pd.read_csv(CSV_PATH)

    def fuse_row(r):
        e, ec, m, mc = r.ear_tag_ocr, r.ear_conf, r.muzzle_id, r.muzzle_conf
        if pd.notna(e) and pd.notna(m) and e == m:
            score = 1 - (1-ec)*(1-mc); return e, score, "agree"
        if pd.notna(e) and pd.notna(m):
            return (e, ec*0.9, "disagree_ear") if ec >= mc else (m, mc*0.9, "disagree_muzzle")
        if pd.notna(e): return e, ec*0.85, "ear_only"
        if pd.notna(m): return m, mc*0.85, "muzzle_only"
        return None, 0.0, "unknown"

    fused = df.apply(lambda r: pd.Series(fuse_row(r), index=["fused_id","fused_conf","mode"]), axis=1)
    df = pd.concat([df, fused], axis=1)
    mask = df["fused_id"].notna()
    print(classification_report(df.loc[mask,"gt_id"], df.loc[mask,"fused_id"], zero_division=0))

    labels = sorted(pd.unique(df[["gt_id","fused_id"]].values.ravel('K')))
    cm = confusion_matrix(df.loc[mask,"gt_id"], df.loc[mask,"fused_id"], labels=labels)
    plt.figure(figsize=(6,6))
    plt.imshow(cm, interpolation='nearest')
    plt.title("Fusion Confusion Matrix"); plt.colorbar()
    plt.xticks(range(len(labels)), labels, rotation=45, ha='right'); plt.yticks(range(len(labels)), labels)
    plt.xlabel("Predicted"); plt.ylabel("Ground Truth"); plt.tight_layout(); plt.show()


⏭️ Provide fused_predictions.csv to compute metrics.


In [ ]:
#@title 📦 Export ONNX (dynamic, half) into Drive
from ultralytics import YOLO
m = YOLO(str(best_w))
onnx_path = PROJ/"best.onnx"
m.export(format="onnx", opset=12, dynamic=True, simplify=True, imgsz=640, half=True)  # produces best.onnx in CWD
# Move it next to the project root if placed elsewhere:
import shutil, os
if not onnx_path.exists() and os.path.exists("/content/best.onnx"):
    shutil.move("/content/best.onnx", onnx_path)
print("✅ ONNX saved at:", onnx_path)
